# Day 3: Classification with K-Nearest Neighbors (Palmer Penguins)

This notebook introduces supervised classification using the K-Nearest Neighbors (KNN) algorithm.

### Motivation

Classification problems require assigning a categorical label to each observation based on its features. KNN is one of the simplest classification algorithms: to classify a new observation, identify the `k` training observations closest to it in feature space, and assign the majority class label among those neighbors.

KNN is an *instance-based* method. There is no model fit in the conventional sense; the training data itself is stored and queried at prediction time. This makes the algorithm conceptually transparent and a useful starting point for understanding more complex classifiers.

**Dataset.** [Palmer Penguins](https://allisonhorst.github.io/palmerpenguins/): 344 penguins from three species (Adelie, Chinstrap, Gentoo) sampled near Palmer Station, Antarctica, with several morphometric measurements per observation. Source: Gorman et al.

**Objective.** Train a KNN classifier to predict species from body measurements, then evaluate on held-out data.

> **Note on notation.** The `k` in KNN refers to the number of *neighbors* used per prediction. It is unrelated to the `k` in K-Means clustering, which refers to the number of *clusters*. Both algorithms use the letter `k` but for different quantities.

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay


### Load the dataset

`seaborn` ships a copy of Palmer Penguins. We drop rows with missing measurements so every subsequent section operates on the same complete-case dataframe.

In [ ]:
df = sns.load_dataset("penguins").dropna().reset_index(drop=True)
print("Shape after dropping NaN:", df.shape)
df.head()


<details>
<summary><strong>Data dictionary</strong> (click to expand)</summary>

<br>

Source: [palmerpenguins package](https://allisonhorst.github.io/palmerpenguins/).

| Column | Meaning |
| --- | --- |
| `species` | Penguin species: Adelie, Chinstrap, or Gentoo |
| `island` | Island where the penguin was observed (Biscoe, Dream, or Torgersen) |
| `bill_length_mm` | Length of the bill, in millimeters |
| `bill_depth_mm` | Depth of the bill, in millimeters |
| `flipper_length_mm` | Length of the flipper, in millimeters |
| `body_mass_g` | Body mass, in grams |
| `sex` | Male or Female |

</details>

### Summary statistics

Species are present in roughly balanced proportions. Each observation has four numeric measurements.

In [ ]:
print("Species distribution:")
print(df["species"].value_counts())

print("\nNumeric feature summary:")
print(df[["bill_length_mm","bill_depth_mm","flipper_length_mm","body_mass_g"]].describe().round(1))


### Visual evidence for class separation

KNN performs well when classes occupy distinct regions of the feature space. The scatter plot below shows bill length on the x-axis and flipper length on the y-axis, colored by species.

Three classes are visually separable in this 2D feature space:

1. Gentoo: long flippers, medium-to-long bills (upper region).
2. Adelie: short bills, short flippers (lower-left).
3. Chinstrap: long bills, short flippers (lower-right).

This separation suggests that KNN should achieve high classification accuracy using these two features alone.

In [ ]:
plt.figure(figsize=(7, 5))
sns.scatterplot(data=df, x="bill_length_mm", y="flipper_length_mm",
                hue="species", style="species", s=70, alpha=0.8)
plt.title("Palmer Penguins: bill length vs flipper length, by species")
plt.xlabel("Bill length (mm)")
plt.ylabel("Flipper length (mm)")
plt.show()


## The KNN algorithm

To classify a new observation, KNN performs three steps:

1. Compute the distance from the new observation to every observation in the training set (typically Euclidean distance).
2. Identify the `k` training observations with the smallest distances.
3. Assign the new observation the most frequent class label among those `k` neighbors.

There is no training procedure in the conventional sense. The fit step stores the training data; all computation happens at prediction time.

The value of `k` controls the smoothness of the decision boundary:

- Small `k` (e.g., 1): boundary closely follows individual training points. High variance, low bias.
- Large `k` (e.g., 25): boundary smooths over local variation. Low variance, higher bias.

We will set `k = 5` as a starting point and address `k` selection in the practice exercises.

### Train-test split and standardization

KNN must be evaluated on data it has not seen during fitting. We hold out 25% of observations as a test set, stratified by species so that all three classes appear in both splits in proportion.

Standardization is essential for KNN: the algorithm uses Euclidean distance, so features with larger numeric ranges dominate the distance metric unless rescaled.

> **New scikit-learn pattern.** In the previous (regression) notebook, you used `model.fit(X, y)` followed by `model.predict(X_test)`. Preprocessing objects such as `StandardScaler` follow a slightly different two-step pattern: `.fit(X)` learns the parameters (here, the mean and standard deviation of each column), and `.transform(X)` applies them to return the rescaled data. The preprocessing object does not produce predictions; it produces a transformed feature matrix.
>
> We call `.fit()` on the training data only, then call `.transform()` on both the training and test sets. This ensures the test set is rescaled using the same parameters as the training set, so no information from the test set influences the scaling. Mixing them (for example, by calling `.fit()` on the full dataset before splitting) is a form of data leakage and biases the test-set evaluation.


In [ ]:
features = ["bill_length_mm", "flipper_length_mm"]
X = df[features].values
y = df["species"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

scaler = StandardScaler().fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training set: {X_train.shape[0]} observations")
print(f"Test set:     {X_test.shape[0]} observations")


### Fit KNN with k = 5

`fit` stores the training data and the corresponding labels. No model parameters are estimated.

In [ ]:
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, y_train)


### Predict and evaluate on the test set

We compute predictions on the held-out test set, then report overall accuracy and the confusion matrix. The confusion matrix indicates which classes are confused with which.

In [ ]:
y_pred = knn.predict(X_test_scaled)
acc = accuracy_score(y_test, y_pred)
print(f"Test accuracy: {acc:.3f}")

cm = confusion_matrix(y_test, y_pred, labels=knn.classes_)
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(cm, display_labels=knn.classes_).plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title("Confusion matrix (test set)")
plt.tight_layout(); plt.show()


### Visualize the decision boundary

Because we used two features, we can plot the regions of feature space that KNN assigns to each class. The plot below shows the decision boundary as colored regions, with the training observations overlaid.

The boundary illustrates how KNN partitions feature space into class regions based on the local distribution of training points.

In [ ]:
x_min, x_max = df["bill_length_mm"].min() - 1, df["bill_length_mm"].max() + 1
y_min, y_max = df["flipper_length_mm"].min() - 1, df["flipper_length_mm"].max() + 1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300),
                     np.linspace(y_min, y_max, 300))
grid = np.c_[xx.ravel(), yy.ravel()]

grid_scaled = scaler.transform(grid)
Z = knn.predict(grid_scaled)

species_to_code = {s: i for i, s in enumerate(knn.classes_)}
Z_codes = np.array([species_to_code[s] for s in Z]).reshape(xx.shape)

fig, ax = plt.subplots(figsize=(7.5, 5.5))
ax.contourf(xx, yy, Z_codes, alpha=0.25, cmap="Set2")
sns.scatterplot(x=X_train[:, 0], y=X_train[:, 1],
                hue=y_train, style=y_train,
                s=60, alpha=0.85, ax=ax)
ax.set_xlabel("Bill length (mm)")
ax.set_ylabel("Flipper length (mm)")
ax.set_title("KNN decision boundary (k = 5), training data overlaid")
plt.tight_layout(); plt.show()


**Result.** KNN with `k = 5` classifies penguins with high test-set accuracy using two standardized body measurements. The decision boundary plot shows three contiguous regions corresponding to the three species, with boundaries placed in low-density gaps between classes.

KNN is a strong baseline for low-dimensional classification problems with clear class structure. It is conceptually transparent, requires no parameter estimation, and produces interpretable decision boundaries. Its main limitations are computational cost at prediction time (every prediction requires distance computations against the full training set) and sensitivity to irrelevant or unscaled features.

### Practice exercises

Complete at least one of (a) and (b).

**(a) Vary `k`.** Refit KNN with `k = 1` and `k = 25`. Plot the decision boundary in each case and compare to the `k = 5` result above. Identify which choice over-fits (boundary follows individual points closely) and which under-fits (boundary ignores local structure).

**(b) Reduce feature informativeness.** Refit KNN using `flipper_length_mm` and `body_mass_g` instead of bill length. Recompute test accuracy and inspect the confusion matrix. Identify which species become confused and infer which feature carried the discriminating signal in the original analysis.

---

**(c) Optional: select `k` from cross-validation.**

The choice of `k` controls the bias-variance tradeoff and should not be selected by inspection of test accuracy (doing so leaks test-set information). The standard approach is k-fold cross-validation on the training set: for each candidate value of `k`, split the training set into folds, fit on the remaining folds, evaluate on the held-out fold, and average. Select the value of `k` that maximizes mean cross-validation accuracy.

Steps:

1. For each candidate `k` in [1, 3, 5, ..., 25], evaluate KNN with `cross_val_score(KNeighborsClassifier(n_neighbors=k), X_train_scaled, y_train, cv=5)`.
2. Plot mean cross-validation accuracy against `k`.
3. Identify the `k` that maximizes the score. Refit on the full training set with this value and report test accuracy.

Worked solutions: `day03_knn_teacher.ipynb`.

In [ ]:
# Complete at least one of (a) and (b). (c) is optional.



## Summary

KNN classifies a new observation by majority vote among its `k` nearest training points in feature space. The standard workflow:

1. Hold out a test set (stratified by class) before any processing.
2. Standardize features using parameters fit on the training set only.
3. Fit `KNeighborsClassifier(n_neighbors=k)` (this stores the training data).
4. Predict on the test set and evaluate with accuracy and confusion matrix.
5. Select `k` via cross-validation on the training set when accuracy is the primary objective.

KNN is appropriate when the feature space is low-dimensional, classes occupy locally distinct regions, and the training set is small enough for prediction-time distance computations to be tractable. It is less appropriate in high-dimensional settings (distances become uniform across observations) or with large training sets where prediction latency matters.